# Magnetic Resonance Imaging - 361.2.6501
## Final Project: Central-Slice Brain Tumor Segmentation

**Students:** Yuval Ratzabi (ID: TODO), Second student (ID: TODO)

This is the complete executable project report using leakage-safe five-fold nested patient-level cross-validation. It performs binary whole-tumor segmentation using only axial slice 80 from T1, T1ce, T2 and FLAIR. Every patient contributes held-out predictions exactly once, allowing all required tables, plots, qualitative examples and failure analyses to be produced from one run.

## Abstract

We compare a classical raw-intensity Gaussian Mixture Model (GMM) baseline with a proposed Boundary + Symmetry model that combines MRI tissue likelihoods, relative brain-boundary position and bilateral asymmetry. Boundary distance, Symmetry and the hierarchy-enhanced Combined model are retained as ablations. All GMM likelihoods are evaluated in log space, and connected posterior components are filtered and expanded using MRI-specific evidence. Performance is estimated with five outer patient-level folds. Within each fold, GMMs are fitted using inner-training patients, parameters are selected only on inner-validation patients, final fold models are refitted on the complete outer-development set, and the outer-test patients are evaluated once.

## 1. Introduction, objective and data description

The objective is binary whole-tumor segmentation from the four BraTS 2020 MRI contrasts. T1 provides anatomical structure, T1ce highlights enhancing tumor, T2 is sensitive to water content, and FLAIR suppresses cerebrospinal fluid while retaining edema-related hyperintensity. The ground-truth BraTS tissue labels are merged into a single tumor mask.

To respect the available computational resources, the same axial slice-selection rule is applied to every patient and every model: only slice 80 is used. Dice and Intersection over Union (IoU) measure spatial overlap. Empty ground truth with an empty prediction is scored as Dice = IoU = 1; missed tumors and false positives on tumor-free slices are therefore also reported explicitly.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from config import *
from data.preprocessing import load_preprocessed_slice
from data.splits import create_nested_cv_folds
from evaluation.cross_validation import evaluate_saved_fold_models, run_nested_cross_validation
from evaluation.optimization import load_selected_parameters
from evaluation.visualizations import (
    choose_qualitative_examples,
    plot_metric_boxplots,
    plot_pipeline_diagnostic,
    plot_qualitative_examples,
    plot_required_scatterplots,
)

%matplotlib inline
sns.set_theme(style="whitegrid")
CV_FIGURES_DIR = Path(CV_OUTPUT_DIR) / "figures"
CV_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

N_BASELINE_TRIALS = 30
N_ADVANCED_TRIALS = 20
FORCE_NEW_FOLDS = False
FORCE_RETRAIN_FOLD_MODELS = False
REUSE_COMPLETED_FOLDS = True

## 2. Leakage-safe nested cross-validation

Five outer folds replace the single fixed train/validation/test split. For each outer fold:

1. The outer-test patients remain untouched.
2. The remaining patients are split into inner training and validation subsets.
3. Fold-specific selection GMMs are fitted only on inner-training patients.
4. Image-processing and probability parameters are selected only on inner validation.
5. Final fold GMMs are refitted on inner train + validation with the selected parameters frozen.
6. The outer-test patients are evaluated once.

The primary comparison is pre-specified as **Raw (4D) versus Boundary + Symmetry**. Combined hierarchy is reported only as a secondary ablation. No outer-test result is used to choose parameters or decide which model is presented as the proposed method.

In [ ]:
folds = create_nested_cv_folds(force=FORCE_NEW_FOLDS)
fold_rows = []
for fold in folds:
    for role in ("train", "validation", "test"):
        manifest = pd.read_csv(CV_SPLITS_DIR / f"fold_{fold['fold']}" / f"{role}_ids.csv")
        positive = manifest["tumor_pixels"] > 0
        fold_rows.append({
            "Fold": fold["fold"],
            "Role": role,
            "Patients": len(manifest),
            "Tumor-present slice 80": int(positive.sum()),
            "Tumor-free slice 80": int((~positive).sum()),
            "Median positive tumor pixels": float(manifest.loc[positive, "tumor_pixels"].median()),
        })
fold_summary = pd.DataFrame(fold_rows)
display(fold_summary)

In [ ]:
outer_manifests = pd.concat([
    pd.read_csv(CV_SPLITS_DIR / f"fold_{fold['fold']}" / "test_ids.csv").assign(fold=fold["fold"])
    for fold in folds
], ignore_index=True)

outer_manifests["tumor_present"] = outer_manifests["tumor_pixels"] > 0
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(data=outer_manifests, x="fold", hue="tumor_present", ax=axes[0])
axes[0].set_title("Tumor-present and tumor-free outer-test slices")
axes[0].set_ylabel("Patients")
axes[0].legend(title="Tumor present")
sns.histplot(
    data=outer_manifests[outer_manifests["tumor_pixels"] > 0],
    x="tumor_pixels", hue="fold", bins=25, element="step", common_norm=False,
    palette="tab10", ax=axes[1],
)
axes[1].set_title("Outer-test tumor-size distribution")
axes[1].set_xlabel("Ground-truth tumor pixels on slice 80")
plt.tight_layout()
plt.savefig(CV_FIGURES_DIR / "outer_fold_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. MRI-specific preprocessing

Each modality is Z-score normalized using only nonzero brain pixels from the same central slice. The 2D brain mask prevents air/background intensities from influencing normalization or sampling. Relative boundary distance is normalized within the brain. Bilateral discrepancy features compare blurred modality intensities with their reflected counterparts along the verified dataset symmetry axis. All preprocessing is identical across folds and models.

In [ ]:
sample_manifest = pd.read_csv(CV_SPLITS_DIR / "fold_1" / "train_ids.csv")
sample_volume = int(sample_manifest.loc[sample_manifest["tumor_pixels"] > 0, "volume_id"].iloc[0])
sample = load_preprocessed_slice(sample_volume)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for index, name in enumerate(MODALITY_NAMES):
    axes.flat[index].imshow(np.where(sample.brain_mask, sample.image[..., index], np.nan), cmap="gray")
    axes.flat[index].set_title(f"{name} - normalized")
axes.flat[4].imshow(sample.brain_mask, cmap="gray")
axes.flat[4].set_title("2D brain mask")
axes.flat[5].imshow(sample.whole_tumor, cmap="gray")
axes.flat[5].set_title("Whole-tumor ground truth")
axes.flat[6].imshow(sample.distance, cmap="viridis")
axes.flat[6].set_title("Relative boundary distance")
axes.flat[7].imshow(np.mean(np.abs(sample.symmetry), axis=-1), cmap="magma")
axes.flat[7].set_title("Mean bilateral discrepancy")
for axis in axes.flat:
    axis.axis("off")
fig.suptitle(f"Fold-1 training volume {sample_volume}, axial slice {SLICE_NUM}")
plt.tight_layout()
plt.savefig(CV_FIGURES_DIR / "preprocessing_example.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Baseline model

The classical baseline uses the four normalized MRI intensities at every brain voxel. Patient-balanced samples fit a 9-component healthy-tissue GMM and separate 4-component GMMs for necrotic/non-enhancing core, edema and enhancing tumor. Stable log-likelihoods and class priors produce a voxelwise tumor posterior. Connected candidate components are then classified using posterior confidence, uncertainty and size, followed by bounded seed expansion.

`T1, T1ce, T2, FLAIR -> patient-balanced GMM fitting -> tumor posterior -> component classification -> bounded expansion -> binary mask`

## 5. Proposed advanced model: Boundary + Symmetry

The proposed model combines two complementary MRI-informed branches. The Boundary branch augments the four contrasts with relative position inside the brain, helping distinguish anatomically implausible edge responses. The Symmetry branch augments the contrasts with four bilateral discrepancy channels, exploiting the approximate symmetry of healthy brain anatomy. Their calibrated tumor probabilities are fused using a validation-selected weight before the same frozen component-processing pipeline is applied.

`boundary-aware GMM posterior + symmetry-aware GMM posterior -> validation-selected weighted fusion -> component classification -> bounded expansion -> binary mask`

The hierarchy-enhanced Combined pipeline is not treated as the proposed model because it was not the strongest previous method. It remains in the evaluation as an ablation of component confirmation and FLAIR/T2-guided expansion.

In [ ]:
model_table = pd.DataFrame([
    {"Model": "Raw (4D)", "Features": "T1, T1ce, T2, FLAIR", "Role": "Required baseline"},
    {"Model": "Boundary distance (5D)", "Features": "MRI + relative boundary distance", "Role": "Ablation"},
    {"Model": "Symmetry (8D)", "Features": "MRI + four bilateral discrepancy channels", "Role": "Ablation"},
    {"Model": "Boundary + Symmetry", "Features": "Weighted fusion of separate boundary and symmetry posteriors", "Role": "Required proposed model"},
    {"Model": "Combined", "Features": "Fusion + optional hierarchy confirmation and FLAIR/T2 expansion", "Role": "Secondary ablation"},
])
display(model_table)

## 6. Fold-specific training and parameter selection

The following cell performs the complete experiment and is the only long-running cell. Every completed fold is saved independently. If execution is interrupted, rerunning with `REUSE_COMPLETED_FOLDS=True` resumes from the first incomplete fold. Fold-specific model directories prevent a GMM fitted for one split from being reused in another.

Run this cell once with the chosen trial counts. Do not change trial counts or regenerate folds after inspecting outer-test results.

In [ ]:
cv_results = run_nested_cross_validation(
    folds,
    n_baseline_trials=N_BASELINE_TRIALS,
    n_advanced_trials=N_ADVANCED_TRIALS,
    force_retrain_models=FORCE_RETRAIN_FOLD_MODELS,
    reuse_completed_folds=REUSE_COMPLETED_FOLDS,
)

### Parameter-selection audit

This table exposes the fold-specific sample counts, hierarchy decision and hierarchy validation gain. The following table shows the selected fusion parameters and confirms that parameters are allowed to vary across inner validation folds while outer-test data remain unused.

In [ ]:
metadata_rows, parameter_rows = [], []
for fold in folds:
    fold_dir = Path(CV_OUTPUT_DIR) / f"fold_{fold['fold']}"
    metadata_rows.append(json.loads((fold_dir / "fold_metadata.json").read_text()))
    selected = load_selected_parameters(fold_dir / "selected_parameters.json")
    image_params = selected["frozen_image_processing_params"]
    fusion_params = selected["model_probability_params"]["Boundary + Symmetry"]
    parameter_rows.append({
        "fold": fold["fold"],
        **{f"image_{key}": value for key, value in image_params.items()},
        **{f"fusion_{key}": value for key, value in fusion_params.items()},
    })
display(pd.DataFrame(metadata_rows).round(4))
display(pd.DataFrame(parameter_rows).round(4))

## 7. Quantitative results

Every patient appears exactly once in `outer_test_per_volume`. The course-required mean and standard deviation are therefore calculated over the pooled out-of-fold patient predictions. Fold-level mean and standard deviation are additionally reported to show stability across data partitions.

In [ ]:
outer_predictions = cv_results["outer_test_per_volume"].copy()
fold_scores = cv_results["outer_test_fold_summaries"].copy()
generalization = cv_results["generalization_summary"].copy()

def summarize_out_of_fold(frame):
    rows = []
    for model_name in MODEL_NAMES:
        group = frame[frame["model"] == model_name]
        tumor = group[group["tumor_present"]]
        rows.append({
            "model": model_name,
            "dice_mean": group["dice"].mean(),
            "dice_std": group["dice"].std(ddof=0),
            "iou_mean": group["iou"].mean(),
            "iou_std": group["iou"].std(ddof=0),
            "tumor_present_dice": tumor["dice"].mean(),
            "precision": tumor["precision"].mean(),
            "recall": tumor["recall"].mean(),
            "missed_tumors": int(group["missed_tumor"].sum()),
            "tumor_slices": int(group["tumor_present"].sum()),
            "empty_slice_false_positives": int(group["empty_slice_false_positive"].sum()),
            "empty_slices": int((~group["tumor_present"]).sum()),
        })
    return pd.DataFrame(rows)

all_model_summary = summarize_out_of_fold(outer_predictions)
all_model_summary["Missed tumors"] = all_model_summary["missed_tumors"].astype(str) + "/" + all_model_summary["tumor_slices"].astype(str)
all_model_summary["Empty-slice FP"] = all_model_summary["empty_slice_false_positives"].astype(str) + "/" + all_model_summary["empty_slices"].astype(str)
display(all_model_summary[[
    "model", "dice_mean", "dice_std", "tumor_present_dice", "precision", "recall",
    "iou_mean", "iou_std", "Missed tumors", "Empty-slice FP",
]].round(4))

### Required baseline-versus-proposed table

The required comparison uses Raw (4D) as the baseline and Boundary + Symmetry as the proposed model.

In [ ]:
PRIMARY_MODELS = ["Raw (4D)", "Boundary + Symmetry"]
required_table = all_model_summary[all_model_summary["model"].isin(PRIMARY_MODELS)][[
    "model", "dice_mean", "dice_std", "iou_mean", "iou_std"
]].rename(columns={
    "model": "Model", "dice_mean": "Dice mean", "dice_std": "Dice std",
    "iou_mean": "IoU mean", "iou_std": "IoU std",
})
required_table["Model"] = pd.Categorical(required_table["Model"], PRIMARY_MODELS, ordered=True)
required_table = required_table.sort_values("Model")
display(required_table.round(4))

### Required Dice and IoU boxplots

In [ ]:
baseline_frame = outer_predictions[outer_predictions["model"] == "Raw (4D)"].copy()
proposed_frame = outer_predictions[outer_predictions["model"] == "Boundary + Symmetry"].copy()
plot_metric_boxplots(
    [baseline_frame, proposed_frame],
    CV_FIGURES_DIR / "required_dice_iou_boxplots.png",
)
plt.show()

### Five-model ablation boxplots

In [ ]:
plot_metric_boxplots(
    [outer_predictions[outer_predictions["model"] == name] for name in MODEL_NAMES],
    CV_FIGURES_DIR / "all_models_dice_iou_boxplots.png",
)
plt.show()

### Required patient-level scatterplots

In [ ]:
plot_required_scatterplots(
    baseline_frame,
    proposed_frame,
    CV_FIGURES_DIR / "required_baseline_proposed_scatterplots.png",
)
plt.show()

### Fold stability and validation-to-outer-test gap

The fold plot shows whether performance is dominated by one favorable partition. The validation-versus-test plot exposes remaining selection optimism without using outer-test scores for tuning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
primary_fold_scores = fold_scores[fold_scores["model"].isin(PRIMARY_MODELS)]
sns.pointplot(data=primary_fold_scores, x="fold", y="dice_mean", hue="model", markers=["o", "s"], ax=axes[0])
axes[0].set_title("Held-out Dice by outer fold")
axes[0].set_ylabel("Mean Dice")

gap_plot = generalization[generalization["model"].isin(PRIMARY_MODELS)].melt(
    id_vars="model",
    value_vars=["validation_dice_mean", "outer_test_dice_mean"],
    var_name="Evaluation", value_name="Mean Dice",
)
sns.barplot(data=gap_plot, x="model", y="Mean Dice", hue="Evaluation", ax=axes[1])
axes[1].set_title("Inner validation versus held-out outer test")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=12)
plt.tight_layout()
plt.savefig(CV_FIGURES_DIR / "fold_stability_and_generalization.png", dpi=300, bbox_inches="tight")
plt.show()
display(generalization.round(4))

### Additional paired baseline-versus-advanced scatterplots

These ablation plots retain the complete paired-model inspection from the original notebook. The primary course comparison remains Raw versus Boundary + Symmetry.

In [ ]:
advanced_names_for_plot = [name for name in MODEL_NAMES if name != "Raw (4D)"]
fig, axes = plt.subplots(2, len(advanced_names_for_plot), figsize=(17, 8))
for column, model_name in enumerate(advanced_names_for_plot):
    advanced_frame = outer_predictions[outer_predictions["model"] == model_name]
    merged = baseline_frame.merge(advanced_frame, on="volume_id", suffixes=("_baseline", "_advanced"))
    for row, (metric, label) in enumerate((("dice", "Dice"), ("iou", "IoU"))):
        x = merged[f"{metric}_baseline"]
        y = merged[f"{metric}_advanced"]
        correlation = np.corrcoef(x, y)[0, 1]
        axes[row, column].scatter(x, y, alpha=0.7)
        axes[row, column].plot([0, 1], [0, 1], "r--", linewidth=1)
        axes[row, column].set(
            xlim=(-0.03, 1.03), ylim=(-0.03, 1.03),
            xlabel=f"Baseline {label}", ylabel=f"{model_name} {label}",
            title=f"r = {correlation:.3f}",
        )
        axes[row, column].grid(alpha=0.25)
fig.suptitle("Paired out-of-fold baseline-versus-advanced comparisons")
plt.tight_layout()
plt.savefig(CV_FIGURES_DIR / "all_paired_model_scatterplots.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Failure analysis and additional diagnostics

In [ ]:
per_volume_comparison = baseline_frame[["volume_id", "fold"]].sort_values("volume_id").reset_index(drop=True)
for model_name in MODEL_NAMES:
    frame = outer_predictions[outer_predictions["model"] == model_name]
    per_volume_comparison = per_volume_comparison.merge(
        frame[["volume_id", "dice", "iou", "precision", "recall", "ground_truth_size", "prediction_size"]].rename(columns={
            column: f"{model_name} {column}" for column in frame.columns if column != "volume_id"
        }),
        on="volume_id",
    )
display(per_volume_comparison.round(4))
per_volume_comparison.to_csv(Path(CV_OUTPUT_DIR) / "all_models_per_volume_wide.csv", index=False)

In [ ]:
tumor_rows = proposed_frame[proposed_frame["tumor_present"]]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(tumor_rows["ground_truth_size"], tumor_rows["dice"], alpha=0.75)
axes[0].set(xlabel="Ground-truth tumor pixels", ylabel="Dice", title="Proposed-model Dice versus tumor size")
axes[1].scatter(tumor_rows["recall"], tumor_rows["precision"], c=tumor_rows["dice"], cmap="viridis", alpha=0.8)
axes[1].set(xlabel="Recall", ylabel="Precision", title="Proposed-model precision-recall pattern")
for axis in axes:
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(CV_FIGURES_DIR / "proposed_failure_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
error_case_table = pd.DataFrame([
    {
        "Model": name,
        "Missed tumor volumes": outer_predictions.loc[
            (outer_predictions["model"] == name) & outer_predictions["missed_tumor"], "volume_id"
        ].astype(int).tolist(),
        "Empty-slice false-positive volumes": outer_predictions.loc[
            (outer_predictions["model"] == name) & outer_predictions["empty_slice_false_positive"], "volume_id"
        ].astype(int).tolist(),
    }
    for name in MODEL_NAMES
])
display(error_case_table)

### Reconstruct held-out details for report figures

Summary CSV files are sufficient for the quantitative results. Detailed posterior and component arrays are reconstructed only for the selected held-out failures and qualitative examples, using the final model and frozen parameters from each patient’s own outer fold.

In [ ]:
selected_examples = choose_qualitative_examples(baseline_frame, proposed_frame)
worst_tumor_volume = int(tumor_rows.sort_values("dice").iloc[0]["volume_id"])
all_missed_volumes = sorted(set(outer_predictions.loc[outer_predictions["missed_tumor"], "volume_id"].astype(int)))
diagnostic_volume_ids = sorted(set(all_missed_volumes + [worst_tumor_volume] + [
    int(value) for value in selected_examples.values() if value is not None
]))

DETAILS = {name: {} for name in MODEL_NAMES}
for fold_index in sorted(set(proposed_frame.loc[proposed_frame["volume_id"].isin(diagnostic_volume_ids), "fold"].astype(int))):
    fold_volume_ids = proposed_frame.loc[
        (proposed_frame["fold"] == fold_index) & proposed_frame["volume_id"].isin(diagnostic_volume_ids),
        "volume_id",
    ].astype(int).tolist()
    fold_details = evaluate_saved_fold_models(folds[fold_index - 1], fold_volume_ids, tuple(MODEL_NAMES))
    for model_name in MODEL_NAMES:
        DETAILS[model_name].update(fold_details[model_name]["details"])

print(f"Reconstructed held-out details for {len(diagnostic_volume_ids)} volumes: {diagnostic_volume_ids}")

### Tumors missed by at least one model

Green denotes true positive, red false positive and blue false negative. Each patient is shown using the model fitted for its own outer fold.

In [ ]:
def error_overlay(ground_truth, prediction):
    overlay = np.zeros((*ground_truth.shape, 4), dtype=float)
    overlay[ground_truth & prediction] = (0.15, 0.90, 0.25, 0.75)
    overlay[~ground_truth & prediction] = (1.00, 0.20, 0.10, 0.75)
    overlay[ground_truth & ~prediction] = (0.10, 0.45, 1.00, 0.85)
    return overlay

missed_rows = []
for volume_id in all_missed_volumes:
    row = {"Volume": volume_id, "GT pixels": DETAILS["Raw (4D)"][volume_id]["ground_truth_size"]}
    for model_name in MODEL_NAMES:
        row[f"{model_name} Dice"] = DETAILS[model_name][volume_id]["dice"]
    missed_rows.append(row)
display(pd.DataFrame(missed_rows).round(3))

rows_per_page = 4
for page, start in enumerate(range(0, len(all_missed_volumes), rows_per_page), start=1):
    page_volumes = all_missed_volumes[start:start + rows_per_page]
    fig, axes = plt.subplots(len(page_volumes), 1 + len(MODEL_NAMES), figsize=(18, 3.2 * len(page_volumes)), squeeze=False)
    for row_index, volume_id in enumerate(page_volumes):
        reference = DETAILS["Raw (4D)"][volume_id]
        flair = reference["data"].image[..., 3]
        ground_truth = reference["data"].whole_tumor
        axes[row_index, 0].imshow(flair, cmap="gray")
        if ground_truth.any():
            axes[row_index, 0].contour(ground_truth, levels=[0.5], colors="yellow", linewidths=1)
        axes[row_index, 0].set_title(f"Volume {volume_id}: FLAIR + GT")
        for column, model_name in enumerate(MODEL_NAMES, start=1):
            details = DETAILS[model_name][volume_id]
            axes[row_index, column].imshow(flair, cmap="gray")
            axes[row_index, column].imshow(error_overlay(ground_truth, details["prediction"]))
            axes[row_index, column].set_title(f"{model_name}\nDice={details['dice']:.3f}")
        for axis in axes[row_index]:
            axis.axis("off")
    plt.tight_layout()
    plt.savefig(CV_FIGURES_DIR / f"missed_tumor_error_maps_{page}.png", dpi=250, bbox_inches="tight")
    plt.show()

### Complete proposed-model pipeline trace

The worst held-out tumor case is traced through posterior formation, component classification and expansion to identify the failure stage.

In [ ]:
worst_details = DETAILS["Boundary + Symmetry"][worst_tumor_volume]
print(f"Worst held-out proposed-model case: volume {worst_tumor_volume}, Dice {worst_details['dice']:.4f}")
plot_pipeline_diagnostic(
    worst_details,
    CV_FIGURES_DIR / f"proposed_pipeline_trace_volume_{worst_tumor_volume}.png",
)
plt.show()
display(pd.DataFrame(worst_details["component_table"]).round(4))

## 9. Required four qualitative examples

Each row contains the four MRI inputs, baseline output, proposed Boundary + Symmetry output and ground truth. The categories are selected deterministically from held-out predictions:

- both models performed well;
- both models performed poorly;
- baseline performed better;
- proposed model performed better.

If no patient satisfies one of the directional categories, the notebook states this explicitly.

In [ ]:
display(pd.DataFrame(selected_examples.items(), columns=["Required category", "Volume"]))
plot_qualitative_examples(
    selected_examples,
    DETAILS["Raw (4D)"],
    DETAILS["Boundary + Symmetry"],
    CV_FIGURES_DIR / "required_qualitative_examples.png",
)
plt.show()

In [ ]:
qualitative_explanations = []
for category, volume_id in selected_examples.items():
    if volume_id is None:
        qualitative_explanations.append({
            "Category": category,
            "Volume": "None",
            "Explanation": "No held-out patient satisfied this requested relationship.",
        })
        continue
    baseline = DETAILS["Raw (4D)"][volume_id]
    proposed = DETAILS["Boundary + Symmetry"][volume_id]
    qualitative_explanations.append({
        "Category": category,
        "Volume": volume_id,
        "Tumor pixels": baseline["ground_truth_size"],
        "Baseline Dice": baseline["dice"],
        "Proposed Dice": proposed["dice"],
        "Proposed precision": proposed["precision"],
        "Proposed recall": proposed["recall"],
        "Explanation": (
            "Interpret using tumor size, modality contrast, posterior support, component acceptance "
            "and expansion shown in the diagnostic figures."
        ),
    })
display(pd.DataFrame(qualitative_explanations).round(4))

### Held-out posterior comparison across all five pipelines

This additional inspection preserves the model-by-model posterior view from the original notebook, now using a patient that was held out from its fold’s complete development set.

In [ ]:
inspection_volume = selected_examples.get("Proposed model performed better") or worst_tumor_volume
fig, axes = plt.subplots(2, 5, figsize=(17, 7))
for column, model_name in enumerate(MODEL_NAMES):
    details = DETAILS[model_name][inspection_volume]
    tumor_probability = details["tumor_posterior"]
    ground_truth = details["data"].whole_tumor
    axes[0, column].imshow(tumor_probability, cmap="magma", vmin=0, vmax=1)
    axes[0, column].set_title(model_name)
    axes[1, column].imshow(details["data"].image[..., 3], cmap="gray")
    axes[1, column].imshow(error_overlay(ground_truth, details["prediction"]))
    axes[1, column].set_title(f"Dice={details['dice']:.3f}")
    axes[0, column].axis("off")
    axes[1, column].axis("off")
axes[0, 0].set_ylabel("Tumor posterior")
axes[1, 0].set_ylabel("FLAIR + TP/FP/FN")
fig.suptitle(f"Held-out posterior comparison - volume {inspection_volume}")
plt.tight_layout()
plt.savefig(CV_FIGURES_DIR / f"posterior_comparison_volume_{inspection_volume}.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Hierarchy ablation

Combined is compared with Boundary + Symmetry only to determine whether component confirmation and FLAIR/T2-guided expansion add value. It is not substituted for the pre-specified proposed model in the required results.

In [ ]:
fusion_ablation = proposed_frame.merge(
    outer_predictions[outer_predictions["model"] == "Combined"],
    on="volume_id", suffixes=("_fusion", "_combined"),
)
hierarchy_selected_folds = int(pd.DataFrame(metadata_rows)["hierarchy_selected"].sum())
ablation_summary = pd.DataFrame([{
    "Hierarchy-selected folds": hierarchy_selected_folds,
    "Mean Combined - Fusion Dice": (fusion_ablation["dice_combined"] - fusion_ablation["dice_fusion"]).mean(),
    "Median Combined - Fusion Dice": (fusion_ablation["dice_combined"] - fusion_ablation["dice_fusion"]).median(),
    "Patients improved by Combined": int((fusion_ablation["dice_combined"] > fusion_ablation["dice_fusion"]).sum()),
    "Patients worsened by Combined": int((fusion_ablation["dice_combined"] < fusion_ablation["dice_fusion"]).sum()),
}])
display(ablation_summary.round(4))

## 11. Conclusions and summary

The automatic summary below reports the primary held-out comparison and the major error counts. Use it together with the plots and qualitative cases to write the final MRI-specific interpretation: relate successes and failures to tumor size, contrast visibility in T1/T1ce/T2/FLAIR, anatomical plausibility, bilateral asymmetry, posterior component rejection and bounded expansion. Discuss central-slice evaluation, limited sample size and classical GMM assumptions as limitations.

In [ ]:
summary_by_model = all_model_summary.set_index("model")
baseline_summary = summary_by_model.loc["Raw (4D)"]
proposed_summary = summary_by_model.loc["Boundary + Symmetry"]
paired_fold_scores = fold_scores.pivot(index="fold", columns="model", values="dice_mean")

print(f"Baseline out-of-fold mean Dice: {baseline_summary['dice_mean']:.4f} ± {baseline_summary['dice_std']:.4f}")
print(f"Proposed out-of-fold mean Dice: {proposed_summary['dice_mean']:.4f} ± {proposed_summary['dice_std']:.4f}")
print(f"Dice change: {proposed_summary['dice_mean'] - baseline_summary['dice_mean']:+.4f}")
print(f"Baseline out-of-fold mean IoU: {baseline_summary['iou_mean']:.4f} ± {baseline_summary['iou_std']:.4f}")
print(f"Proposed out-of-fold mean IoU: {proposed_summary['iou_mean']:.4f} ± {proposed_summary['iou_std']:.4f}")
print(f"IoU change: {proposed_summary['iou_mean'] - baseline_summary['iou_mean']:+.4f}")
print(f"Outer folds won by proposed model: {(paired_fold_scores['Boundary + Symmetry'] > paired_fold_scores['Raw (4D)']).sum()}/{len(folds)}")
print(f"Baseline missed tumors: {int(baseline_summary['missed_tumors'])}/{int(baseline_summary['tumor_slices'])}")
print(f"Proposed missed tumors: {int(proposed_summary['missed_tumors'])}/{int(proposed_summary['tumor_slices'])}")
print(f"Baseline empty-slice false positives: {int(baseline_summary['empty_slice_false_positives'])}/{int(baseline_summary['empty_slices'])}")
print(f"Proposed empty-slice false positives: {int(proposed_summary['empty_slice_false_positives'])}/{int(proposed_summary['empty_slices'])}")